### ***SUMMARY***

Framing divides each audio signal into short, fixed-size sections that overlap. Speech changes over time, but it is nearly stable within each short frame. Feature methods such as magnitude spectra, MFCC, LPCC, and PLP can therefore describe each frame consistently. Fixed-size frames also give the classifier samples with the same dimensions.

The saved NumPy archive keeps the frames, frame numbers, encoded path IDs, and path lookup table in separate arrays. Each path ID refers to one relative path in the lookup table. This format stores each path once, uses only numeric frame data, and does not require pickle. Later steps can reconstruct the time order, examine neighboring frames, and trace a result to its source file. The source-file link also supports labels and prevents data leakage because frames from one recording must stay in the same data group.

Saving the framed data makes preprocessing repeatable and avoids framing every file again during feature extraction. Later notebooks can load one consistent data set, apply windowing and feature extraction, and compare methods on the same frames.

In [ ]:
from pathlib import Path

# In a notebook, Path.cwd() is the folder the notebook runs from.
NOTEBOOK_DIR:Path = Path.cwd()
REPOSITORY_ROOT:Path = NOTEBOOK_DIR

DIR_SOURCE:Path = REPOSITORY_ROOT / "Artists"
DIR_OUTPUT:Path = NOTEBOOK_DIR / "preprocessing-01-framing.npz"

print(f"Source directory: {DIR_SOURCE}")
print(f"Output file: {DIR_OUTPUT}")

### ***GETTING FILE DIRECTORIES***

In [ ]:
AUDIO_SUFFIXES:set[str] = {".flac", ".m4a", ".mp3", ".ogg", ".wav"}

source_dir:Path = DIR_SOURCE.resolve()
if not source_dir.is_dir():
    raise FileNotFoundError(f"Source directory does not exist: {source_dir}")
FILES_AUDIO:list[Path] = sorted(
    path
    for path in source_dir.rglob("*")
    if path.is_file() and path.suffix.lower() in AUDIO_SUFFIXES
)
if not FILES_AUDIO:
    raise FileNotFoundError(f"No supported audio FILES_AUDIO found under {source_dir}")

for x in FILES_AUDIO:
    print(x)

### ***FRAMING THE AUDIO FILES***

In [ ]:
import librosa
import numpy as np

SAMPLE_RATE:int = 16_000
# 0.025 seconds
FRAME_SIZE:int = 400       
# 0.010 seconds
STEP_SIZE:int = 160         

def frame_audio(file_audio: Path) -> np.ndarray:
    audio:np.ndarray
    audio, _ = librosa.load(file_audio, sr=SAMPLE_RATE, mono=True, dtype=np.float32)

    if audio.size < FRAME_SIZE:
        return np.empty((0, FRAME_SIZE), dtype=np.float32)

    # sliding window view creates a reference to the original audio 
    # on frame[i] it returns audio [i to i+FRAME_SIZE]
    # [::STEP_SIZE] is native python functionality which extracts only very other entry in a list
    # ::STEP_SIZE maps 0->0, 1->STEP_SIZE*1, 2->STEP_SIZE*2 ... i -> STEP_SIZE*i
    # so frames[1] = sliding_window[STEP_SIZE] = audio[STEP_SIZE to STEP_SIZE + FRAME_SIZE]
    frames = np.lib.stride_tricks.sliding_window_view(audio, FRAME_SIZE)[::STEP_SIZE]
    return frames.astype(np.float32, copy=True)

# print(FILES_AUDIO[0])
# for x in (frame_audio(FILES_AUDIO[0])):
#     print(x) 

In [ ]:
frame_blocks:list[np.ndarray] = []
frame_number_blocks:list[np.ndarray] = []
path_id_blocks:list[np.ndarray] = []

# Store each path once as a path relative to the source directory.
PATHS:np.ndarray = np.asarray(
    [audio_path.relative_to(source_dir).as_posix() for audio_path in FILES_AUDIO]
)

for path_id, audio_dir in enumerate(FILES_AUDIO):
    frames:np.ndarray = frame_audio(audio_dir)
    frame_blocks.append(frames)
    frame_number_blocks.append(
        np.arange(1, len(frames) + 1, dtype=np.uint32)
    )
    path_id_blocks.append(np.full(len(frames), path_id, dtype=np.uint32))
    print(f"{audio_dir}: {len(frames)} frames")


ALL_AUDIO_FRAMES:np.ndarray = np.concatenate(frame_blocks, axis=0).astype(
    np.float32, copy=False
)
FRAME_NUMBERS:np.ndarray = np.concatenate(frame_number_blocks)
PATH_IDS:np.ndarray = np.concatenate(path_id_blocks)

### ***SAVING THE AUDIO FRAME FILES***

In [ ]:
if DIR_OUTPUT.suffix.lower() != ".npz":
    raise ValueError(f"Output file must use the .npz extension: {DIR_OUTPUT}")

DIR_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
np.savez(
    DIR_OUTPUT,
    frames=ALL_AUDIO_FRAMES,
    frame_numbers=FRAME_NUMBERS,
    path_ids=PATH_IDS,
    paths=PATHS,
)

print(f"Saved {len(ALL_AUDIO_FRAMES)} frames to {DIR_OUTPUT}")
print("Members: frames (float32), frame_numbers (uint32), path_ids (uint32), paths")

### ***VERIFYING THE SAVED AUDIO FRAMES***

In [ ]:
with np.load(DIR_OUTPUT, allow_pickle=False) as saved:
    loaded_frames:np.ndarray = saved["frames"]
    loaded_frame_numbers:np.ndarray = saved["frame_numbers"]
    loaded_path_ids:np.ndarray = saved["path_ids"]
    loaded_paths:np.ndarray = saved["paths"]

assert loaded_frames.dtype == np.float32
assert loaded_frame_numbers.dtype == np.uint32
assert loaded_path_ids.dtype == np.uint32
assert loaded_paths.dtype.kind == "U"
assert np.array_equal(loaded_frames, ALL_AUDIO_FRAMES), (
    "One or more audio frames changed after saving."
)
assert np.array_equal(loaded_frame_numbers, FRAME_NUMBERS), (
    "One or more frame numbers changed after saving."
)
assert np.array_equal(loaded_path_ids, PATH_IDS), (
    "One or more encoded path IDs changed after saving."
)
assert np.array_equal(loaded_paths, PATHS), (
    "One or more paths changed after saving."
)
assert len(loaded_frames) == len(loaded_frame_numbers) == len(loaded_path_ids)
assert np.all(loaded_path_ids < len(loaded_paths)), "A path ID is out of range."

for path_id, relative_path in enumerate(loaded_paths):
    frame_numbers = loaded_frame_numbers[loaded_path_ids == path_id]
    assert np.array_equal(
        frame_numbers, np.arange(1, len(frame_numbers) + 1, dtype=np.uint32)
    ), f"Frame sequence is not continuous for {relative_path}"

print(
    f"Integrity verified: {len(loaded_frames)} frames and "
    f"{len(loaded_paths)} encoded paths match the arrays in memory."
)